In [ ]:
import os

import torch
from torch import nn, Tensor
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from torchvision.transforms import v2
from torch.backends import cudnn
from torch import GradScaler
from torch import optim
from tqdm import tqdm
import numpy as np
import pickle

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")
enable_half = device.type != "cpu"
scaler = GradScaler(device, enabled=enable_half)

print("Grad scaler is enabled:", enable_half)
device

In [ ]:
if os.path.exists("/kaggle/input") and os.path.exists("/kaggle/working"):
    print("Running on Kaggle.")
    SVHN_test = "/kaggle/input/fii-atnn-2025-competition-2/SVHN_test.pkl"
    SVHN_train = "/kaggle/input/fii-atnn-2025-competition-2/SVHN_train.pkl"
else:
    print("Not on Kaggle.")
    SVHN_test = "data/SVHN_test.pkl"
    SVHN_train = "data/SVHN_train.pkl"

In [ ]:
class SVHN_Dataset(Dataset):
    def __init__(self, train: bool, transforms: v2.Transform):
        path = SVHN_test
        if train:
            path = SVHN_train
        with open(path, "rb") as fd:
            self.data = pickle.load(fd)

        self.transforms = transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i: int):
        image, label = self.data[i]
        if self.transforms is None:
            return image, label
        return self.transforms(image), label

In [ ]:
# basic_transforms = v2.Compose([
#     v2.ToImage(),
#     v2.ToDtype(torch.float32, scale=True),
#     v2.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25), inplace=True)
# ])

# train_set = SVHN_Dataset(train=True, transforms=basic_transforms)
# test_set = SVHN_Dataset(train=False, transforms=basic_transforms)

# train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
# test_loader = DataLoader(test_set, batch_size=500)

In [ ]:
class VGG13(nn.Module):
    def __init__(self):
        super(VGG13, self).__init__()

        self.layers = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Classifier
            nn.Flatten(),
            nn.Linear(512, 100)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


In [ ]:
# model = VGG13().to(device)
# model = torch.jit.script(model)
# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
# optimizer = optim.SGD(model.parameters(), lr=0.003, fused=True, momentum = 0.9, weight_decay=1e-3)

In [ ]:
# # CONFIGURATION A — BASELINE

# best_acc = 0
# epochs = 20
# for epoch in range(epochs):
#     model.train()
#     correct, total, total_loss = 0, 0, 0
#     for inputs, targets in train_loader:
#         inputs, targets = inputs.to(device), targets.to(device)
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = criterion(outputs, targets)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#         correct += (outputs.argmax(1) == targets).sum().item()
#         total += targets.size(0)
#     acc = 100 * correct / total
#     print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.2f}%")
#     if acc > best_acc:
#         best_acc = acc
#         torch.save(model.state_dict(), "/kaggle/working/best_model_A.pth")

# print(f"Best Train Accuracy (A): {best_acc:.2f}%")

In [ ]:
# def train():
#     model.train()
#     correct = 0
#     total = 0
    
#     for inputs, targets in train_loader:
#         inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
#         with torch.autocast(device.type, enabled=enable_half):
#             outputs = model(inputs)
#             loss = criterion(outputs, targets)
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         optimizer.zero_grad()

#         predicted = outputs.argmax(1)
#         total += targets.size(0)
#         correct += predicted.eq(targets).sum().item()
    
#     return 100.0 * correct / total

In [ ]:
# @torch.inference_mode()
# def inference():
#     model.eval()
    
#     labels = []
    
#     for inputs, _ in test_loader:
#         inputs = inputs.to(device, non_blocking=True)
#         with torch.autocast(device.type, enabled=enable_half):
#             outputs = model(inputs)

#         predicted = outputs.argmax(1).tolist()
#         labels.extend(predicted)
    
#     return labels

In [ ]:
# best = 0.0
# best_epoch = 0
# epochs = list(range(5))

# with tqdm(epochs) as tbar:
#     for epoch in tbar:
#         train_acc = train()

#         if train_acc > best:
#             best = train_acc
#             best_epoch = epoch

#         tbar.set_description(f"Train: {train_acc:.2f}, Best: {best:.2f} at epoch {best_epoch}")

In [ ]:
# # CONFIGURATION B — Light Augmentation + Adam

# light_aug = v2.Compose([
#     v2.ToImage(),
#     v2.RandomHorizontalFlip(p=0.5),
#     v2.RandomCrop((32, 32), padding=4),
#     v2.ToDtype(torch.float32, scale=True),
#     v2.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25))
# ])

# basic_transforms = v2.Compose([
#     v2.ToImage(),
#     v2.ToDtype(torch.float32, scale=True),
#     v2.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25), inplace=True)
# ])

# train_loader = DataLoader(SVHN_Dataset(train=True, transforms=light_aug), batch_size=64, shuffle=True)
# test_loader = DataLoader(SVHN_Dataset(train=False, transforms=basic_transforms), batch_size=500)

# model = VGG13().to(device)
# model = torch.jit.script(model)
# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
# optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# best_acc = 0
# epochs = 30
# for epoch in range(epochs):
#     model.train()
#     correct, total, total_loss = 0, 0, 0
#     for inputs, targets in train_loader:
#         inputs, targets = inputs.to(device), targets.to(device)
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = criterion(outputs, targets)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#         correct += (outputs.argmax(1) == targets).sum().item()
#         total += targets.size(0)
#     acc = 100 * correct / total
#     print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.2f}%")
#     if acc > best_acc:
#         best_acc = acc
#         torch.save(model.state_dict(), "/kaggle/working/best_model_B.pth")

# print(f"Best Train Accuracy (B): {best_acc:.2f}%")

# @torch.inference_mode()
# def inference():
#     model.eval()
    
#     labels = []
    
#     for inputs, _ in test_loader:
#         inputs = inputs.to(device, non_blocking=True)
#         with torch.autocast(device.type, enabled=enable_half):
#             outputs = model(inputs)

#         predicted = outputs.argmax(1).tolist()
#         labels.extend(predicted)
    
#     return labels

In [ ]:
# #Configuration C — Strong Augmentation + Label Smoothing + Cosine LR

# strong_aug = v2.Compose([
#     v2.ToImage(),
#     v2.RandomHorizontalFlip(p=0.5),
#     v2.RandomRotation(10),
#     v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
#     v2.ToDtype(torch.float32, scale=True),
#     v2.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25))
# ])

# basic_transforms = v2.Compose([
#     v2.ToImage(),
#     v2.ToDtype(torch.float32, scale=True),
#     v2.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25), inplace=True)
# ])

# train_loader = DataLoader(SVHN_Dataset(train=True, transforms=strong_aug), batch_size=128, shuffle=True)
# test_loader = DataLoader(SVHN_Dataset(train=False, transforms=basic_transforms), batch_size=500)

# model = VGG13().to(device)
# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
# optimizer = optim.SGD(model.parameters(), lr=0.02, momentum=0.9, weight_decay=5e-4)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

# best_acc = 0
# epochs = 25
# for epoch in range(epochs):
#     model.train()
#     correct, total, total_loss = 0, 0, 0
#     for inputs, targets in train_loader:
#         inputs, targets = inputs.to(device), targets.to(device)
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = criterion(outputs, targets)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#         correct += (outputs.argmax(1) == targets).sum().item()
#         total += targets.size(0)
#     scheduler.step()
#     acc = 100 * correct / total
#     print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.2f}%")
#     if acc > best_acc:
#         best_acc = acc
#         torch.save(model.state_dict(), "/kaggle/working/best_model_C.pth")

# print(f"Best Train Accuracy (C): {best_acc:.2f}%")

# @torch.inference_mode()
# def inference():
#     model.eval()
    
#     labels = []
    
#     for inputs, _ in test_loader:
#         inputs = inputs.to(device, non_blocking=True)
#         with torch.autocast(device.type, enabled=enable_half):
#             outputs = model(inputs)

#         predicted = outputs.argmax(1).tolist()
#         labels.extend(predicted)
    
#     return labels

In [ ]:
#Configuration D — Test-Time Augmentation (TTA)

strong_aug = v2.Compose([
    v2.ToImage(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(10),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25))
])

basic_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.25, 0.25, 0.25), inplace=True)
])

train_loader = DataLoader(SVHN_Dataset(train=True, transforms=strong_aug), batch_size=128, shuffle=True)
test_loader = DataLoader(SVHN_Dataset(train=False, transforms=basic_transforms), batch_size=500)

model = VGG13().to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.30)
optimizer = optim.SGD(model.parameters(), lr=0.015, momentum=0.9, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

best_acc = 0
epochs = 30
for epoch in range(epochs):
    model.train()
    correct, total, total_loss = 0, 0, 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == targets).sum().item()
        total += targets.size(0)
    scheduler.step()
    acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.2f}%")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "/kaggle/working/best_model_D.pth")

print(f"Best Train Accuracy (D): {best_acc:.2f}%")


@torch.inference_mode()
def inference():
    model.eval()
    
    labels = []
    
    for inputs, _ in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        with torch.autocast(device.type, enabled=enable_half):
            outputs = model(inputs)
            outputs += model(torch.flip(inputs, dims=[-1]))          # horizontal
            outputs += model(torch.rot90(inputs, 1, dims=[2, 3]))    # 90°
            outputs /= 3.0

        predicted = outputs.argmax(1).tolist()
        labels.extend(predicted)
    
    return labels

In [ ]:
data = {
    "ID": [],
    "target": []
}


for i, label in enumerate(inference()):
    data["ID"].append(i)
    data["target"].append(label)

df = pd.DataFrame(data)
df.to_csv("/kaggle/working/submission.csv", index=False)